# 从零实现 R-GCN + DistMult：关系消息、时序负采样与 Filtered Ranking

本 Notebook 用 PyTorch 基础张量手写带 basis decomposition 的 `RGCNLayer`、`RGCNEncoder` 与 `DistMultDecoder`，不使用 PyG、DGL、现成 GNN/KG 层。重点不是“在小数据上记住三元组”，而是把有向关系、逆关系、自环、按关系归一化、类型/时间感知负采样、无泄漏消息图、filtered MRR/Hits 以及实体/关系词表制品绑定全部落实成可执行断言。

数据为固定种子的虚构知识图谱，CPU 单线程离线运行；排名结果只证明受控 fixture 的链路正确，不代表真实 KG 的泛化效果。

In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

from dataclasses import dataclass
import hashlib
import json
import math
import random
import time

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 3801
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")
DTYPE = torch.float32

def canonical_hash(payload) -> str:
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]

assert DEVICE.type == "cpu" and torch.get_num_threads() == 1
assert torch.initial_seed() == SEED
assert not any(name in globals() for name in ("torch_geometric", "dgl"))

## 1. 实体、关系、类型与时间合同

图谱含 12 个 `person`、2 个 `group`、2 个当前 `topic`，另有 1 个时间 5 才激活的未来 topic 作为负采样边界探针。三种正向关系为：`member_of(person,group)`、`prefers(group,topic)`、`likes(person,topic)`。实体 `active_from` 决定某个时间点是否允许参与负采样；domain/range 决定候选实体类型。

事实按时间严格切分：时间 1–2 是 train，时间 3 是 validation，时间 4 是 test。单条三元组是 `(head_id, relation_id, tail_id)`，批量打分输入为 `[B,3]`；消息图使用 `edge_index:[2,E_msg]`、`edge_type:[E_msg]`，节点表示沿网络保持 `[N,D]`。所有 membership 和 group preference 都在训练快照；部分 person 的 `likes` 留作未来链接。验证/测试事实绝不能进入 R-GCN 消息图，否则即使不读取标签，边本身已经泄漏答案。

In [ ]:
people = [f"person:{i}" for i in range(12)]
groups = ["group:0", "group:1"]
topics = ["topic:0", "topic:1", "topic:future"]
entity_names = people + groups + topics
entity_to_id = {name: i for i, name in enumerate(entity_names)}
entity_type = {name: name.split(":", 1)[0] for name in entity_names}
active_from = {name: 0 for name in entity_names}
active_from["topic:future"] = 5

relation_names = ["member_of", "prefers", "likes"]
relation_to_id = {name: i for i, name in enumerate(relation_names)}
domain = {"member_of": "person", "prefers": "group", "likes": "person"}
range_type = {"member_of": "group", "prefers": "topic", "likes": "topic"}

@dataclass(frozen=True)
class Fact:
    head: str
    relation: str
    tail: str
    timestamp: int
    split: str

facts = []
for i, person in enumerate(people):
    facts.append(Fact(person, "member_of", groups[i % 2], 1, "train"))
facts += [Fact(groups[0], "prefers", topics[0], 1, "train"),
          Fact(groups[1], "prefers", topics[1], 1, "train")]
for i, person in enumerate(people):
    split, ts = ("train", 2) if i < 8 else (("val", 3) if i < 10 else ("test", 4))
    facts.append(Fact(person, "likes", topics[i % 2], ts, split))

def validate_fact_at_time(fact: Fact, *, cutoff: int | None = None,
                          allowed_splits: frozenset[str] | None = None) -> None:
    if fact.head not in entity_to_id or fact.tail not in entity_to_id or fact.relation not in relation_to_id:
        raise ValueError("事实包含冻结词表之外的实体或关系")
    if not isinstance(fact.timestamp, int) or fact.timestamp < 0:
        raise ValueError("事实 timestamp 必须是非负整数")
    if entity_type[fact.head] != domain[fact.relation] or entity_type[fact.tail] != range_type[fact.relation]:
        raise ValueError("事实违反 relation domain/range")
    if active_from[fact.head] > fact.timestamp or active_from[fact.tail] > fact.timestamp:
        raise ValueError("事实引用了查询时间尚未激活的实体")
    if cutoff is not None and fact.timestamp > cutoff:
        raise ValueError("事实时间超过消息快照 cutoff")
    if allowed_splits is not None and fact.split not in allowed_splits:
        raise ValueError("事实 split 不属于消息快照允许集合")


def encode_fact(fact: Fact) -> tuple[int, int, int]:
    validate_fact_at_time(fact)
    return entity_to_id[fact.head], relation_to_id[fact.relation], entity_to_id[fact.tail]

split_facts = {s: [f for f in facts if f.split == s] for s in ("train", "val", "test")}
encoded_all = {encode_fact(f) for f in facts}
assert tuple(len(split_facts[s]) for s in ("train", "val", "test")) == (22, 2, 2)
assert max(f.timestamp for f in split_facts["train"]) < min(f.timestamp for f in split_facts["val"])
assert max(f.timestamp for f in split_facts["val"]) < min(f.timestamp for f in split_facts["test"])
assert len(encoded_all) == len(facts)
assert all(entity_type[f.head] == domain[f.relation] and entity_type[f.tail] == range_type[f.relation] for f in facts)

try:
    encode_fact(Fact("person:0", "likes", "topic:future", 1, "train"))
    raise AssertionError("未激活实体进入事实表后未被拒绝")
except ValueError as exc:
    assert "尚未激活" in str(exc)


## 2. 正向、逆向与自环是三种不同语义

原始事实 `(h,r,t)` 产生正向消息边 `h → t`，关系编号 `r`；为让 head 也读取邻域，再显式增加 `t → h`，其关系是独立的 `r_inverse`，不能复用同一权重。总消息关系数因此是 `2R`。

自环不伪装成又一种知识关系，而由每层独立矩阵 $W_0$ 处理：

$$h_i' = W_0h_i + \sum_r\sum_{j\in\mathcal N_i^r}\frac{1}{c_{i,r}}W_rh_j + b,$$

其中 $c_{i,r}=|\mathcal N_i^r|$。R-GCN 消息图只由显式 `cutoff` 之前、允许 split 内的事实构建；builder 返回包含 cutoff、split 与事实指纹的不可变快照。

In [ ]:
NUM_BASE_RELATIONS = len(relation_names)
NUM_MESSAGE_RELATIONS = 2 * NUM_BASE_RELATIONS


@dataclass(frozen=True)
class MessageGraphSnapshot:
    edge_index: torch.Tensor
    edge_type: torch.Tensor
    cutoff: int
    allowed_splits: frozenset[str]
    snapshot_hash: str


def build_message_graph(source_facts: list[Fact], *, cutoff: int,
                        allowed_splits: frozenset[str]) -> MessageGraphSnapshot:
    if not source_facts:
        raise ValueError("消息快照不能没有事实")
    if not isinstance(cutoff, int) or cutoff < 0 or not allowed_splits:
        raise ValueError("cutoff/allowed_splits 合同非法")
    edges, edge_types, payload, seen = [], [], [], set()
    for fact in source_facts:
        validate_fact_at_time(fact, cutoff=cutoff, allowed_splits=allowed_splits)
        h, r, t = encode_fact(fact)
        fact_key = (h, r, t, fact.timestamp, fact.split)
        if fact_key in seen:
            raise ValueError("消息快照包含重复事实")
        seen.add(fact_key)
        payload.append(fact_key)
        edges.extend([(h, t), (t, h)])
        edge_types.extend([r, r + NUM_BASE_RELATIONS])
    order = sorted(range(len(edges)), key=lambda i: (edges[i][1], edge_types[i], edges[i][0]))
    edge_index = torch.tensor([edges[i] for i in order], dtype=torch.long).T.contiguous()
    edge_type = torch.tensor([edge_types[i] for i in order], dtype=torch.long)
    snapshot_hash = canonical_hash({"cutoff": cutoff, "splits": sorted(allowed_splits),
                                    "facts": sorted(payload)})
    return MessageGraphSnapshot(edge_index, edge_type, cutoff, allowed_splits, snapshot_hash)


message_snapshot = build_message_graph(
    split_facts["train"], cutoff=2, allowed_splits=frozenset({"train"})
)
message_edges, message_types = message_snapshot.edge_index, message_snapshot.edge_type
message_pairs = {(int(message_edges[0, i]), int(message_types[i]), int(message_edges[1, i]))
                 for i in range(message_edges.shape[1])}
heldout = {encode_fact(f) for f in split_facts["val"] + split_facts["test"]}
assert message_snapshot.cutoff == 2 and message_snapshot.allowed_splits == frozenset({"train"})
assert len(message_snapshot.snapshot_hash) == 20
assert message_edges.shape == (2, 44) and message_types.shape == (44,)
assert all((h, r, t) not in message_pairs for h, r, t in heldout)
assert all((t, r + NUM_BASE_RELATIONS, h) not in message_pairs for h, r, t in heldout)
assert set(message_types.tolist()) == set(range(NUM_MESSAGE_RELATIONS))
assert not any(int(message_edges[0, i]) == int(message_edges[1, i]) for i in range(message_edges.shape[1]))

try:
    build_message_graph(split_facts["val"], cutoff=2, allowed_splits=frozenset({"train"}))
    raise AssertionError("held-out 事实进入 train 快照后未被拒绝")
except ValueError as exc:
    assert "cutoff" in str(exc) or "split" in str(exc)


## 3. 按 `(target, relation)` 归一化的精确 oracle

同一 target 接受不同关系时，分母不能混在一起。令 `key = target * num_relations + relation`，对 key 做计数即可得到每条边的 $1/c_{i,r}$。

Oracle 中，关系 0 有 `0→2`、`1→2` 两条入边，所以各为 1/2；关系 1 只有 `0→1`，权重为 1。这个测试会抓住按 source 度数、按 target 总度数或忘记方向的实现。

In [ ]:
def relation_target_norm(edge_index: torch.Tensor, edge_type: torch.Tensor,
                         num_nodes: int, num_relations: int) -> torch.Tensor:
    if edge_index.ndim != 2 or edge_index.shape[0] != 2 or edge_type.ndim != 1 or edge_type.numel() != edge_index.shape[1]:
        raise ValueError("edge_index/edge_type shape 不匹配")
    if num_nodes <= 0 or num_relations <= 0:
        raise ValueError("节点数和关系数必须为正")
    if edge_index.numel() and (int(edge_index.min()) < 0 or int(edge_index.max()) >= num_nodes):
        raise ValueError("边端点越界")
    if edge_type.numel() and (int(edge_type.min()) < 0 or int(edge_type.max()) >= num_relations):
        raise ValueError("关系编号越界")
    target = edge_index[1]
    key = target * num_relations + edge_type
    counts = torch.bincount(key, minlength=num_nodes * num_relations)
    return counts[key].to(DTYPE).reciprocal()

oracle_edge_index = torch.tensor([[0, 1, 0], [2, 2, 1]], dtype=torch.long)
oracle_edge_type = torch.tensor([0, 0, 1], dtype=torch.long)
oracle_norm = relation_target_norm(oracle_edge_index, oracle_edge_type, 3, 2)
assert torch.equal(oracle_norm, torch.tensor([0.5, 0.5, 1.0]))
assert math.isclose(float(oracle_norm[:2].sum()), 1.0, abs_tol=1e-7)
assert math.isclose(float(oracle_norm[2]), 1.0, abs_tol=1e-7)

## 4. Basis decomposition 与 `RGCNLayer`

若每个关系都有完整矩阵，参数为 $R F_{in}F_{out}$，关系多时很昂贵。Basis decomposition 写成

$$W_r=\sum_{b=1}^{B}a_{rb}V_b,$$

参数降为 $BF_{in}F_{out}+RB$。`bases:(B,F_in,F_out)` 与 `coefficients:(R,B)` 都是可训练参数。前向先选出每条边对应的 $W_r$，计算 source 消息，再按 target 聚合；self-loop 矩阵单独相加。

In [ ]:
class RGCNLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int, num_relations: int, num_bases: int):
        super().__init__()
        if min(in_features, out_features, num_relations, num_bases) <= 0 or num_bases > num_relations:
            raise ValueError("R-GCN 维度或 basis 数非法")
        self.in_features, self.out_features = in_features, out_features
        self.num_relations, self.num_bases = num_relations, num_bases
        self.bases = nn.Parameter(torch.empty(num_bases, in_features, out_features))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        self.self_weight = nn.Parameter(torch.empty(in_features, out_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
        nn.init.xavier_uniform_(self.bases)
        nn.init.xavier_uniform_(self.coefficients)
        nn.init.xavier_uniform_(self.self_weight)

    def relation_weights(self) -> torch.Tensor:
        return torch.einsum("rb,bio->rio", self.coefficients, self.bases)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_type: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2 or x.shape[1] != self.in_features or not torch.isfinite(x).all():
            raise ValueError("x shape 或数值非法")
        norm = relation_target_norm(edge_index, edge_type, x.shape[0], self.num_relations).to(x.device)
        source, target = edge_index
        weights = self.relation_weights()
        messages = torch.einsum("ei,eio->eo", x[source], weights[edge_type])
        messages = messages * norm[:, None]
        out = x.new_zeros((x.shape[0], self.out_features))
        out.index_add_(0, target, messages)
        return out + x @ self.self_weight + self.bias

oracle_layer = RGCNLayer(2, 2, 2, 2)
with torch.no_grad():
    oracle_layer.bases[0].copy_(torch.eye(2))
    oracle_layer.bases[1].copy_(torch.tensor([[0., 1.], [1., 0.]]))
    oracle_layer.coefficients.copy_(torch.eye(2))
    oracle_layer.self_weight.zero_(); oracle_layer.bias.zero_()
oracle_x = torch.tensor([[1., 0.], [0., 2.], [3., 4.]])
oracle_out = oracle_layer(oracle_x, oracle_edge_index, oracle_edge_type)
oracle_expected = torch.tensor([[0., 0.], [0., 1.], [0.5, 1.]])
assert torch.allclose(oracle_out, oracle_expected, atol=1e-7)
assert oracle_layer.relation_weights().shape == (2, 2, 2)
assert sum(p.numel() for p in oracle_layer.parameters()) == 2*2*2 + 2*2 + 2*2 + 2

# 无边图把关系消息完全拿掉，单独验证 self-loop 矩阵与 bias。
self_only = RGCNLayer(2, 2, 1, 1)
with torch.no_grad():
    self_only.bases.zero_(); self_only.coefficients.zero_()
    self_only.self_weight.copy_(torch.tensor([[2., 0.], [0., 3.]]))
    self_only.bias.copy_(torch.tensor([1., -1.]))
self_x = torch.tensor([[1., 2.], [3., 4.]])
self_out = self_only(self_x, torch.empty((2, 0), dtype=torch.long), torch.empty(0, dtype=torch.long))
assert torch.equal(self_out, torch.tensor([[3., 5.], [7., 11.]]))


## 5. `RGCNEncoder` 与 `DistMultDecoder`

Encoder 从实体 ID embedding 开始，经两层 R-GCN 得到 $z_e\in\mathbb R^d$。DistMult 为每个**正向业务关系**学习向量 $w_r$：

$$s(h,r,t)=\langle z_h,w_r,z_t\rangle=\sum_k z_{h,k}w_{r,k}z_{t,k}.$$

DistMult 的形式对交换 head/tail 是对称的，因此不适合表达所有反对称关系；这里用严格 domain/range 与 R-GCN 上下文做受控教学，生产选型需比较 ComplEx、RotatE 等。逆关系只服务 encoder 消息传播，decoder 仍使用原始三种关系词表。

In [ ]:
class RGCNEncoder(nn.Module):
    def __init__(self, num_entities: int, hidden_dim: int, num_message_relations: int, num_bases: int = 3):
        super().__init__()
        self.num_entities = num_entities
        self.entity_embedding = nn.Embedding(num_entities, hidden_dim)
        self.layer1 = RGCNLayer(hidden_dim, hidden_dim, num_message_relations, num_bases)
        self.layer2 = RGCNLayer(hidden_dim, hidden_dim, num_message_relations, num_bases)
        nn.init.xavier_uniform_(self.entity_embedding.weight)

    def forward(self, edge_index: torch.Tensor, edge_type: torch.Tensor) -> torch.Tensor:
        entity_ids = torch.arange(self.num_entities, device=self.entity_embedding.weight.device)
        h = F.relu(self.layer1(self.entity_embedding(entity_ids), edge_index, edge_type))
        return self.layer2(h, edge_index, edge_type)

class DistMultDecoder(nn.Module):
    def __init__(self, num_relations: int, hidden_dim: int):
        super().__init__()
        self.num_relations = num_relations
        self.relation_embedding = nn.Embedding(num_relations, hidden_dim)
        nn.init.xavier_uniform_(self.relation_embedding.weight)

    def forward(self, z: torch.Tensor, heads: torch.Tensor,
                relations: torch.Tensor, tails: torch.Tensor) -> torch.Tensor:
        if not (heads.shape == relations.shape == tails.shape) or heads.ndim != 1:
            raise ValueError("三元组索引 shape 必须一致且为一维")
        if relations.numel() and (int(relations.min()) < 0 or int(relations.max()) >= self.num_relations):
            raise ValueError("decoder 关系编号越界")
        return (z[heads] * self.relation_embedding(relations) * z[tails]).sum(dim=-1)

class RGCNDistMult(nn.Module):
    def __init__(self, num_entities: int, num_relations: int, hidden_dim: int = 24):
        super().__init__()
        self.encoder = RGCNEncoder(num_entities, hidden_dim, 2 * num_relations, num_bases=3)
        self.decoder = DistMultDecoder(num_relations, hidden_dim)

    def encode(self, edge_index: torch.Tensor, edge_type: torch.Tensor) -> torch.Tensor:
        return self.encoder(edge_index, edge_type)

    def score(self, z: torch.Tensor, triples: torch.Tensor) -> torch.Tensor:
        if triples.ndim != 2 or triples.shape[1] != 3:
            raise ValueError("triples shape 必须为 (K,3)")
        return self.decoder(z, triples[:, 0], triples[:, 1], triples[:, 2])

    def forward(self, edge_index: torch.Tensor, edge_type: torch.Tensor,
                triples: torch.Tensor) -> torch.Tensor:
        return self.score(self.encode(edge_index, edge_type), triples)

model_probe = RGCNDistMult(len(entity_names), len(relation_names), hidden_dim=12)
z_probe = model_probe.encode(message_edges, message_types)
probe_triple = torch.tensor([encode_fact(facts[0])])
score_probe = model_probe.score(z_probe, probe_triple)
assert z_probe.shape == (17, 12) and score_probe.shape == (1,)
assert torch.isfinite(z_probe).all() and torch.isfinite(score_probe).all()
assert torch.allclose(score_probe, model_probe(message_edges, message_types, probe_triple), atol=1e-7)

## 6. 类型/时间感知负采样

随机从全部实体替换 tail 会制造大量一眼可辨的假负例，例如 `person member_of topic`，模型只需学类型而不是关系。`sample_tail_negatives` 只从关系 range 类型、且在查询时间已激活的实体中选择，并排除当前训练快照已知真事实。

训练时排除集合只能来自 train snapshot；若用未来 validation/test 事实指导训练负采样，虽然“避免了假负例”，却向训练过程泄露了未来知识。评估按每个查询的 timestamp 构造 `truth_as_of`；未来 validation/test 事实既不能指导训练负采样，也不能改变更早时点的 validation rank。

In [ ]:
def candidate_ids(relation: str, timestamp: int, side: str = "tail") -> list[int]:
    if relation not in relation_to_id or side not in {"head", "tail"}:
        raise ValueError("关系或采样侧非法")
    if not isinstance(timestamp, int) or timestamp < 0:
        raise ValueError("timestamp 必须是非负整数")
    wanted = range_type[relation] if side == "tail" else domain[relation]
    return [entity_to_id[name] for name in entity_names
            if entity_type[name] == wanted and active_from[name] <= timestamp]


train_known = {encode_fact(f) for f in split_facts["train"]}
assert entity_to_id["topic:future"] not in candidate_ids("likes", 4, "tail")
assert entity_to_id["topic:future"] in candidate_ids("likes", 5, "tail")


def sample_tail_negatives(source_facts: list[Fact], known_snapshot: set[tuple[int, int, int]],
                          repeats: int = 1) -> torch.Tensor:
    if repeats <= 0:
        raise ValueError("repeats 必须为正")
    negatives = []
    for fact in source_facts:
        validate_fact_at_time(fact)
        h, r, true_t = encode_fact(fact)
        allowed = candidate_ids(fact.relation, fact.timestamp, "tail")
        valid = [t for t in allowed if t != true_t and (h, r, t) not in known_snapshot]
        if len(valid) < repeats:
            raise ValueError("合法唯一负样本不足；拒绝重复放大或退化为错类型实体")
        start = (h + r) % len(valid)
        ordered = valid[start:] + valid[:start]
        negatives.extend((h, r, t) for t in ordered[:repeats])
    return torch.tensor(negatives, dtype=torch.long)


train_positive = torch.tensor([encode_fact(f) for f in split_facts["train"]], dtype=torch.long)
train_negative = sample_tail_negatives(split_facts["train"], train_known, repeats=1)
assert train_positive.shape == train_negative.shape == (22, 3)
assert torch.unique(train_negative, dim=0).shape[0] == train_negative.shape[0]
assert not any(tuple(row) in train_known for row in train_negative.tolist())
for h, r, t in train_negative.tolist():
    rel = relation_names[r]
    assert entity_type[entity_names[h]] == domain[rel]
    assert entity_type[entity_names[t]] == range_type[rel]

try:
    sample_tail_negatives([split_facts["train"][0]], train_known, repeats=2)
    raise AssertionError("候选不足时重复负例未被拒绝")
except ValueError as exc:
    assert "唯一负样本不足" in str(exc)


## 7. Filtered MRR 与 Hits@K

对查询 `(h,r,?)`，只在符合 range 类型和查询时间的候选 tail 上排名。若其他候选在查询 timestamp 之前已经是真答案，应在排名时过滤，但保留当前 target；未来才成立的事实不能参与当前过滤。令目标名次为 $rank_q$：

$$MRR=\frac1Q\sum_q\frac1{rank_q},\qquad Hits@K=\frac1Q\sum_q\mathbf 1[rank_q\le K].$$

这里使用“严格大于目标分数”的数量加 1；生产评估还应固定 tie policy，并分别报告 raw/filtered、head/tail prediction 与关系分桶指标。

In [ ]:
def truth_as_of(benchmark_facts: list[Fact], timestamp: int) -> set[tuple[int, int, int]]:
    if not isinstance(timestamp, int) or timestamp < 0:
        raise ValueError("timestamp 必须是非负整数")
    truths = set()
    for fact in benchmark_facts:
        validate_fact_at_time(fact)
        if fact.timestamp <= timestamp:
            truths.add(encode_fact(fact))
    return truths


def filtered_ranks(z: torch.Tensor, decoder: DistMultDecoder, queries: list[Fact],
                   benchmark_facts: list[Fact]) -> torch.Tensor:
    if not queries:
        raise ValueError("queries 不能为空")
    ranks = []
    for fact in queries:
        validate_fact_at_time(fact)
        h, r, target = encode_fact(fact)
        all_true = truth_as_of(benchmark_facts, fact.timestamp)
        if (h, r, target) not in all_true:
            raise ValueError("查询 target 不在该时点真值快照中")
        candidates = candidate_ids(fact.relation, fact.timestamp, "tail")
        if target not in candidates:
            raise ValueError("查询 target 在该时点不属于合法候选")
        kept = [candidate for candidate in candidates
                if candidate == target or (h, r, candidate) not in all_true]
        triples = torch.tensor([(h, r, candidate) for candidate in kept], dtype=torch.long)
        scores = decoder(z, triples[:, 0], triples[:, 1], triples[:, 2])
        target_pos = kept.index(target)
        rank = 1 + int((scores > scores[target_pos]).sum().item())
        ranks.append(rank)
    return torch.tensor(ranks, dtype=torch.long)


def ranking_metrics(ranks: torch.Tensor) -> dict[str, float]:
    if ranks.ndim != 1 or ranks.numel() == 0 or (ranks < 1).any():
        raise ValueError("ranks 必须是非空正整数向量")
    return {"MRR": float((1.0 / ranks.float()).mean()),
            "Hits@1": float((ranks <= 1).float().mean()),
            "Hits@3": float((ranks <= 3).float().mean())}


# 真正调用 filtered_ranks：过去的另一个真答案应过滤，未来答案不得改变 validation rank。
toy_h = entity_to_id["person:0"]
toy_r = relation_to_id["likes"]
toy_t0 = entity_to_id["topic:0"]
toy_t1 = entity_to_id["topic:1"]
toy_z = torch.zeros(len(entity_names), 1)
toy_z[toy_h] = 1.0; toy_z[toy_t0] = 1.0; toy_z[toy_t1] = 2.0
toy_decoder = DistMultDecoder(len(relation_names), 1)
with torch.no_grad():
    toy_decoder.relation_embedding.weight.fill_(1.0)
val_query = Fact("person:0", "likes", "topic:0", 3, "val")
past_alternative = Fact("person:0", "likes", "topic:1", 2, "train")
future_alternative = Fact("person:0", "likes", "topic:1", 4, "test")
raw_rank = filtered_ranks(toy_z, toy_decoder, [val_query], [val_query])
past_filtered_rank = filtered_ranks(toy_z, toy_decoder, [val_query], [val_query, past_alternative])
future_safe_rank = filtered_ranks(toy_z, toy_decoder, [val_query], [val_query, future_alternative])
assert int(raw_rank[0]) == 2 and int(past_filtered_rank[0]) == 1
assert torch.equal(future_safe_rank, raw_rank)
assert ranking_metrics(torch.tensor([1, 2])) == {"MRR": 0.75, "Hits@1": 0.5, "Hits@3": 1.0}


## 8. 训练、验证选型与一次性测试

每轮在固定 train message graph 上编码实体，对正例最小化 `softplus(-score)`、对类型合法负例最小化 `softplus(score)`。validation 的 filtered MRR 只用于保存 checkpoint；test 在加载最佳 checkpoint 后报告一次。固定负例让演示可复现，真实训练通常应在每轮重采样并记录 sampler 版本。

In [ ]:
torch.manual_seed(SEED + 1)
model = RGCNDistMult(len(entity_names), len(relation_names), hidden_dim=24).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.018, weight_decay=1e-5)
initial_loss = None
best_mrr, best_state = -1.0, None
train_start = time.perf_counter()
for epoch in range(121):
    model.train(); optimizer.zero_grad(set_to_none=True)
    z = model.encode(message_edges, message_types)
    pos_scores = model.score(z, train_positive)
    neg_scores = model.score(z, train_negative)
    loss = F.softplus(-pos_scores).mean() + F.softplus(neg_scores).mean()
    if initial_loss is None:
        initial_loss = float(loss.detach())
    loss.backward()
    if epoch == 0:
        grads = [model.encoder.entity_embedding.weight.grad, model.encoder.layer1.bases.grad,
                 model.encoder.layer1.coefficients.grad, model.encoder.layer1.self_weight.grad,
                 model.decoder.relation_embedding.weight.grad]
        assert all(g is not None and torch.isfinite(g).all() and float(g.abs().sum()) > 0 for g in grads)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    optimizer.step()
    if epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
            val_z = model.encode(message_edges, message_types)
            val_ranks = filtered_ranks(val_z, model.decoder, split_facts["val"], facts)
            val_mrr = ranking_metrics(val_ranks)["MRR"]
        if val_mrr > best_mrr:
            best_mrr = val_mrr
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

assert best_state is not None and initial_loss is not None
model.load_state_dict(best_state); model.eval()
with torch.no_grad():
    final_z = model.encode(message_edges, message_types)
    final_loss = float(F.softplus(-model.score(final_z, train_positive)).mean()
                       + F.softplus(model.score(final_z, train_negative)).mean())
    test_ranks = filtered_ranks(final_z, model.decoder, split_facts["test"], facts)
    test_metrics = ranking_metrics(test_ranks)
train_seconds = time.perf_counter() - train_start
assert final_loss < initial_loss * 0.45
assert best_mrr >= 0.75 and test_metrics["MRR"] >= 0.75
assert test_metrics["Hits@3"] == 1.0 and train_seconds < 12.0
print({"initial_loss": round(initial_loss, 4), "final_loss": round(final_loss, 4),
       "val_mrr": best_mrr, "test": test_metrics, "seconds": round(train_seconds, 3)})

## 9. 失败模式、复杂度与生产边界

- **边泄漏**：先把全量图编码再切 train/test，会直接把未来链接放入消息传播。必须冻结 train snapshot 后再构图。
- **逆关系复用权重**：`member_of` 与 `member_of_inverse` 方向语义不同；共用编号会降低表达力并掩盖方向 bug。
- **把 self-loop 当业务事实**：self-loop 是网络更新项，不应进入 KG 事实表或排名真值。
- **错类型负例**：指标看似很好，模型其实只学会 schema。应记录候选生成器、时间截点和 rejection rate。
- **Filtered 指标污染**：时序评估只能使用查询 timestamp 之前的真值过滤；训练 sampler 只读 train snapshot，validation 也不能查看未来 test 真值。
- **DistMult 对称性**：对反对称、多对多复杂关系需比较更合适 decoder。

单层稀疏消息约为 $O(EF_{in}F_{out})$（实现可通过先按关系分组优化），basis 合成约为 $O(RBF_{in}F_{out})$；全候选排名成本随实体数线性增长。生产需要分块打分、近似召回、分类型校准和新实体冷启动策略。

In [ ]:
def state_hash(module: nn.Module) -> str:
    digest = hashlib.sha256()
    for name, tensor in sorted(module.state_dict().items()):
        digest.update(name.encode("utf-8")); digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()[:20]

entity_vocab_hash = canonical_hash({name: entity_to_id[name] for name in entity_names})
relation_vocab_hash = canonical_hash({"forward": {name: relation_to_id[name] for name in relation_names},
                                      "inverse": {name + "_inverse": relation_to_id[name] + NUM_BASE_RELATIONS for name in relation_names}})
type_schema_hash = canonical_hash({"entity_type": entity_type, "domain": domain, "range": range_type})
train_snapshot_hash = message_snapshot.snapshot_hash
artifact = {
    "architecture": "RGCN-basis2layer-DistMult-v1",
    "seed": SEED,
    "entity_vocab_hash": entity_vocab_hash,
    "relation_vocab_hash": relation_vocab_hash,
    "type_schema_hash": type_schema_hash,
    "train_snapshot_hash": train_snapshot_hash,
    "message_cutoff": message_snapshot.cutoff,
    "state_hash": state_hash(model),
    "test_metrics": test_metrics,
}
artifact["artifact_id"] = canonical_hash(artifact)

def score_named_triple(head: str, relation: str, tail: str, expected_artifact_id: str) -> float:
    if expected_artifact_id != artifact["artifact_id"]:
        raise RuntimeError("artifact 绑定不一致")
    if head not in entity_to_id or tail not in entity_to_id or relation not in relation_to_id:
        raise KeyError("实体或关系不在冻结词表")
    fact = Fact(head, relation, tail, 4, "serve")
    triple = torch.tensor([encode_fact(fact)], dtype=torch.long)
    with torch.no_grad():
        z = model.encode(message_edges, message_types)
        return float(model.score(z, triple).item())

served = score_named_triple("person:10", "likes", "topic:0", artifact["artifact_id"])
assert math.isfinite(served)
assert len({entity_vocab_hash, relation_vocab_hash, type_schema_hash, train_snapshot_hash,
            artifact["state_hash"]}) == 5
try:
    score_named_triple("person:10", "likes", "topic:0", "forged")
    raise AssertionError("伪造 artifact 未被拒绝")
except RuntimeError as exc:
    assert "artifact" in str(exc)
print({"artifact_id": artifact["artifact_id"], "entity_vocab_hash": entity_vocab_hash,
       "relation_vocab_hash": relation_vocab_hash})

try:
    score_named_triple("person:10", "likes", "topic:future", artifact["artifact_id"])
    raise AssertionError("服务在时间 4 接受了时间 5 才激活的实体")
except ValueError as exc:
    assert "尚未激活" in str(exc)


## 10. 面试复盘与论文来源

回答 R-GCN/KG 工程题时，应从关系方向与快照边界开始，再讲 basis decomposition、decoder、负采样候选空间和 filtered ranking。若只展示一个训练 loss，而没有证明 validation/test 边未进入 encoder，就无法说明评估可信。

主要来源：Schlichtkrull et al., [**Modeling Relational Data with Graph Convolutional Networks**](https://arxiv.org/abs/1703.06103), ESWC 2018；Yang et al., [**Embedding Entities and Relations for Learning and Inference in Knowledge Bases**](https://arxiv.org/abs/1412.6575)（DistMult）, ICLR 2015。此处是教学规模复现，并未复制论文完整数据处理与超参。